In [8]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Linear Model Libraries
from scipy.stats import skew, kurtosis
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from itertools import product

print("numpy:", np.__version__)

# Compatibility aliases for loading pickles created with NumPy 2.x
import numpy.core
import numpy.core.numeric
import numpy.core.multiarray
import numpy.core.umath

sys.modules["numpy._core"] = numpy.core
sys.modules["numpy._core.numeric"] = numpy.core.numeric
sys.modules["numpy._core.multiarray"] = numpy.core.multiarray
sys.modules["numpy._core.umath"] = numpy.core.umath

numpy: 1.26.4


In [4]:
bat_df = pd.read_pickle("data/bat_df.pkl")
print(bat_df.shape)

(124, 12)


In [5]:
# Splits from original paper
numBat1 = 41
numBat2 = 43
numBat3 = 40
numBat = numBat1 + numBat2 + numBat3

test_ind = np.hstack((np.arange(0, (numBat1 + numBat2), 2), 83))
train_ind = np.arange(1, (numBat1 + numBat2 - 1), 2)
secondary_test_ind = np.arange(numBat - numBat3, numBat)

train_df = bat_df.iloc[train_ind].reset_index(drop=True)
primary_test_df = bat_df.iloc[test_ind].reset_index(drop=True)
secondary_test_df = bat_df.iloc[secondary_test_ind].reset_index(drop=True)

print("train_df:", train_df.shape)
print("primary_test_df:", primary_test_df.shape)
print("secondary_test_df:", secondary_test_df.shape)

train_df: (41, 12)
primary_test_df: (43, 12)
secondary_test_df: (40, 12)


In [6]:
# GET 20 MODEL FEATURES
all_feature_rows = []

# Loop through each training/test set
for split_name, df in [
    ("train", train_df),
    ("primary_test", primary_test_df),
    ("secondary_test", secondary_test_df)
]:

    # Loop through each battery
    for row_idx, row in df.iterrows():

        # Get cycle 100 - 10 discharge
        q10 = np.asarray(row["cycles"]["9"]["Qdlin"], dtype=float).squeeze()
        q100 = np.asarray(row["cycles"]["99"]["Qdlin"], dtype=float).squeeze()
        dq = q100 - q10

        # Get top-level battery arrays
        QD = np.asarray(row["QD"], dtype=float).squeeze()
        chargetime = np.asarray(row["chargetime"], dtype=float).squeeze()
        Tmax = np.asarray(row["Tmax"], dtype=float).squeeze()
        Tmin = np.asarray(row["Tmin"], dtype=float).squeeze()
        Tavg = np.asarray(row["Tavg"], dtype=float).squeeze()
        IR = np.asarray(row["IR"], dtype=float).squeeze()

        # Get slope and intercept of discharge capacity (cycle 2 to 100)
        x_2_100 = np.arange(2, 101)    # cycles 2 to 100
        y_2_100 = QD[1:100]            # cycle 2 to 100 discharge capacity
        slope_2_100, intercept_2_100 = np.polyfit(x_2_100, y_2_100, 1)

        # Get slope and intercept of discharge capacity (cycle 91 to 100)
        x_91_100 = np.arange(91, 101)  # cycles 91 to 100
        y_91_100 = QD[90:100]          # cycle 91 to 100 discharge capacity
        slope_91_100, intercept_91_100 = np.polyfit(x_91_100, y_91_100, 1)

        # Get integral of temperature over time, cycles 2 to 100
        temp_integral_2_100 = 0

        # Cycle 2 to 100
        for cyc_idx in range(1, 100):
            cyc = row["cycles"][str(cyc_idx)]
            T_cycle = np.asarray(cyc["T"], dtype=float).squeeze()
            t_cycle = np.asarray(cyc["t"], dtype=float).squeeze()
            temp_integral_2_100 += np.trapz(T_cycle, t_cycle)

        # Remove absolute max from discharge capacity
        QD_first_100 = QD[:100].copy()
        max_idx = np.argmax(QD_first_100)
        QD_without_max = np.delete(QD_first_100, max_idx)
        QD_max_minus_2 = np.max(QD_without_max) - QD[1]

        # Model features
        feature_row = {
            "split": split_name,
            "battery_id": row["battery_id"],
            "cycle_life": row["cycle_life"],

            # 1. Min: change of discharge capacity 100 - 10
            "dq_min_100_10": np.log10(np.abs(np.min(dq))),

            # 2. Mean: change of discharge capacity 100 - 10
            "dq_mean_100_10": np.log10(np.abs(np.mean(dq))),

            # 3. Variance: change of discharge capacity 100 - 10 (use sample variance from paper)
            "dq_var_100_10": np.log10(np.abs(np.var(dq, ddof=1))),

            # 4. Skewness: change of discharge capacity 100 - 10
            "dq_skew_100_10": np.log10(np.abs(skew(dq))),

            # 5. Kurtosis: change of discharge capacity 100 - 10
            "dq_kurt_100_10": np.log10(np.abs(kurtosis(dq))),

            # 6. Value at 2V: change of discharge capacity 100 - 10
            "dq_2volt_100_10": np.log10(np.abs(dq[-1])),

            # 7. Slope of linear fit to capacity fade curve, cycles 2 to 100
            "QD_slope_2_100": slope_2_100,

            # 8. Intercept of linear fit to capacity fade curve, cycles 2 to 100
            "QD_intercept_2_100": intercept_2_100,

            # 9. Slope of linear fit to capacity fade curve, cycles 91 to 100
            "QD_slope_91_100": slope_91_100,

            # 10. Intercept of linear fit to capacity fade curve, cycles 91 to 100
            "QD_intercept_91_100": intercept_91_100,

            # 11. Discharge capacity, cycle 2
            "QD_cycle_2": QD[1],

            # 12. Difference between max discharge capacity and discharge capacity, cycle 2
            "QD_max_minus_2": QD_max_minus_2,

            # 13. Discharge capacity, cycle 100
            "QD_cycle_100": QD[99],

            # 14. Average charge time, first 5 cycles (cycle 2 to 6 in paper)
            "avg_chargetime_first_5": np.mean(chargetime[1:6]),

            # 15. Max temperature, cycles 2 to 100
            "max_temp_2_100": np.max(Tmax[1:100]),

            # 16. Min temperature, cycles 2 to 100
            "min_temp_2_100": np.min(Tmin[1:100]),

            # 17. Integral of temperature over time, cycles 2 to 100
            "temp_integral_2_100": temp_integral_2_100,

            # 18. Internal resistance, cycle 2
            "IR_cycle_2": IR[1],

            # 19. Minimum internal resistance, cycles 2 to 100
            "IR_min_2_100": np.min(IR[1:100]),

            # 20. Internal resistance, difference between cycle 100 and cycle 2
            "IR_minus_100_2": IR[99] - IR[1],
        }

        all_feature_rows.append(feature_row)

features_df = pd.DataFrame(all_feature_rows)
features_df = features_df.replace([np.inf, -np.inf], np.nan)

display(features_df.head())
print("features_df shape:", features_df.shape)

,split,battery_id,cycle_life,dq_min_100_10,dq_mean_100_10,dq_var_100_10,dq_skew_100_10,dq_kurt_100_10,dq_2volt_100_10,QD_slope_2_100,...,QD_cycle_2,QD_max_minus_2,QD_cycle_100,avg_chargetime_first_5,max_temp_2_100,min_temp_2_100,temp_integral_2_100,IR_cycle_2,IR_min_2_100,IR_minus_100_2
0,train,b1c1,2160.0,-1.958457,-2.387257,-5.013525,-0.367163,0.012464,-2.922940,0.000006,...,1.075301,0.007000,1.080630,13.409150,34.712265,29.230637,180165.946144,0.017039,0.000000,-0.000042
1,train,b1c3,1434.0,-1.722149,-2.127507,-4.442179,-0.357486,0.039579,-2.754730,0.000017,...,1.079723,0.006461,1.084750,12.025140,31.691414,29.023619,169351.876369,0.016370,0.000000,0.000039
2,train,b1c5,1074.0,-1.598965,-1.955699,-4.178443,-0.825794,0.089031,-2.411109,0.000011,...,1.076127,0.005916,1.079779,10.967850,34.654667,28.914812,171164.585651,0.016438,0.015923,-0.000112
3,train,b1c7,870.0,-1.417557,-1.778697,-3.813052,-0.477013,0.049426,-2.135501,-0.000006,...,1.093864,0.004040,1.095762,10.025082,36.053928,29.368095,173739.065551,0.016311,0.000000,-0.000038
4,train,b1c11,788.0,-1.625407,-2.076849,-4.146462,-0.225594,0.081187,-3.235201,0.000022,...,1.053779,0.007549,1.059972,11.668876,37.128967,29.877810,183165.811556,0.016575,0.000000,-0.000304


features_df shape: (124, 23)


In [7]:
feature_cols = [
    "dq_min_100_10",
    "dq_mean_100_10",
    "dq_var_100_10",
    "dq_skew_100_10",
    "dq_kurt_100_10",
    "dq_2volt_100_10",
    "QD_slope_2_100",
    "QD_intercept_2_100",
    "QD_slope_91_100",
    "QD_intercept_91_100",
    "QD_cycle_2",
    "QD_max_minus_2",
    "QD_cycle_100",
    "avg_chargetime_first_5",
    "max_temp_2_100",
    "min_temp_2_100",
    "temp_integral_2_100",
    "IR_cycle_2",
    "IR_min_2_100",
    "IR_minus_100_2",
]

train_features_df = features_df[features_df["split"] == "train"].reset_index(drop=True)
primary_test_features_df = features_df[features_df["split"] == "primary_test"].reset_index(drop=True)
secondary_test_features_df = features_df[features_df["split"] == "secondary_test"].reset_index(drop=True)

X_train = train_features_df[feature_cols]
X_primary_test = primary_test_features_df[feature_cols]
X_secondary_test = secondary_test_features_df[feature_cols]

y_train = np.log10(train_features_df["cycle_life"].astype(float).values)
y_primary_test = np.log10(primary_test_features_df["cycle_life"].astype(float).values)
y_secondary_test = np.log10(secondary_test_features_df["cycle_life"].astype(float).values)

print("\nShapes:")
print("X_train:", X_train.shape)
print("X_primary_test:", X_primary_test.shape)
print("X_secondary_test:", X_secondary_test.shape)
print("y_train:", y_train.shape)
print("y_primary_test:", y_primary_test.shape)
print("y_secondary_test:", y_secondary_test.shape)


Shapes:
X_train: (41, 20)
X_primary_test: (43, 20)
X_secondary_test: (40, 20)
y_train: (41,)
y_primary_test: (43,)
y_secondary_test: (40,)


In [ ]:
# Tune Elastic Net hyperparameters
alphas = np.logspace(-5, 0, 100)

l1_ratios = np.array([
    0.0001, 0.0005, 0.001, 0.005
])

# Use 4-fold CV to evaluate each hyperparameter combination (same as original paper)
kf = KFold(n_splits=4, shuffle=True, random_state=0)

rows = []

for alpha, l1_ratio in product(alphas, l1_ratios):

    fold_rmse = []
    fold_mpe = []
    fold_nonzero = []

    for train_idx, val_idx in kf.split(X_train):

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("enet", ElasticNet(
                alpha=alpha,
                l1_ratio=l1_ratio,
                max_iter=1_000_000,
                tol=1e-5,
                random_state=0
            ))
        ])

        model.fit(X_train.iloc[train_idx], y_train[train_idx])

        pred_cycles = 10 ** model.predict(X_train.iloc[val_idx])
        true_cycles = 10 ** y_train[val_idx]

        fold_rmse.append(np.sqrt(mean_squared_error(true_cycles, pred_cycles)))
        fold_mpe.append(np.mean(np.abs(pred_cycles - true_cycles) / true_cycles) * 100)
        fold_nonzero.append(np.sum(model.named_steps["enet"].coef_ != 0))

    # Fit on full training data to count final selected features
    final_tmp_model = Pipeline([
        ("scaler", StandardScaler()),
        ("enet", ElasticNet(
            alpha=alpha,
            l1_ratio=l1_ratio,
            max_iter=1_000_000,
            tol=1e-5,
            random_state=0
        ))
    ])

    final_tmp_model.fit(X_train, y_train)

    final_nonzero = np.sum(final_tmp_model.named_steps["enet"].coef_ != 0)

    rows.append({
        "alpha": alpha,
        "l1_ratio": l1_ratio,
        "cv_rmse": np.mean(fold_rmse),
        "cv_mpe": np.mean(fold_mpe),
        "mean_nonzero_features": np.mean(fold_nonzero),
        "final_nonzero_features": final_nonzero
    })

cv_results = pd.DataFrame(rows).sort_values("cv_mpe").reset_index(drop=True)

print("Best Elastic Net models:")
display(cv_results.head(20))

Best Ridge models:


,alpha,cv_rmse,cv_mpe
0,0.322863,95.848120,8.710117
1,0.338110,95.890364,8.711474
2,0.354077,95.938449,8.713156
3,0.370799,95.992487,8.716303
4,0.308303,95.811597,8.719797
5,0.388310,96.052582,8.720844
6,0.406648,96.118825,8.725542
7,0.294400,95.780672,8.729629
8,0.425852,96.191296,8.730393
9,0.281124,95.755214,8.739606


In [ ]:
# Pick best hyperparameters from CV table
best_alpha = cv_results.loc[0, "alpha"]
best_l1_ratio = cv_results.loc[0, "l1_ratio"]

print("Best alpha:", best_alpha)
print("Best l1_ratio:", best_l1_ratio)
print("Validation CV RMSE:", cv_results.loc[0, "cv_rmse"])
print("Validation CV percent error:", cv_results.loc[0, "cv_mpe"])
print("Mean nonzero features:", cv_results.loc[0, "mean_nonzero_features"])
print("Final nonzero features:", cv_results.loc[0, "final_nonzero_features"])

# Train final Elastic Net model on full training set
final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("enet", ElasticNet(
        alpha=best_alpha,
        l1_ratio=best_l1_ratio,
        max_iter=1_000_000,
        tol=1e-5,
        random_state=0
    ))
])

final_model.fit(X_train, y_train)

# Train
train_pred = 10 ** final_model.predict(X_train)
train_true = 10 ** y_train

train_rmse = np.sqrt(mean_squared_error(train_true, train_pred))
train_mpe = np.mean(np.abs(train_pred - train_true) / train_true) * 100

# Primary test
primary_pred = 10 ** final_model.predict(X_primary_test)
primary_true = 10 ** y_primary_test

primary_rmse = np.sqrt(mean_squared_error(primary_true, primary_pred))
primary_mpe = np.mean(np.abs(primary_pred - primary_true) / primary_true) * 100

# Secondary test
secondary_pred = 10 ** final_model.predict(X_secondary_test)
secondary_true = 10 ** y_secondary_test

secondary_rmse = np.sqrt(mean_squared_error(secondary_true, secondary_pred))
secondary_mpe = np.mean(np.abs(secondary_pred - secondary_true) / secondary_true) * 100

summary_df = pd.DataFrame({
    "split": [
        "validation_cv",
        "train",
        "primary_test",
        "secondary_test"
    ],
    "rmse_cycles": [
        cv_results.loc[0, "cv_rmse"],
        train_rmse,
        primary_rmse,
        secondary_rmse
    ],
    "mean_percent_error": [
        cv_results.loc[0, "cv_mpe"],
        train_mpe,
        primary_mpe,
        secondary_mpe
    ]
})

display(summary_df)

Best alpha: 0.3228628202536727
Validation CV RMSE: 95.84811982742175
Validation CV percent error: 8.710116735392727


,split,rmse_cycles,mean_percent_error
0,validation_cv,95.848120,8.710117
1,train,43.442367,4.675787
2,primary_test,273.039740,18.689264
3,secondary_test,190.884786,13.705855
